In [ ]:
!pip install pandas numpy scikit-learn torch torchvision torchaudio

^C


  Using cached pandas-2.3.3-cp39-cp39-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl.metadata (59 kB)
  Using cached scikit_learn-1.6.1-cp39-cp39-win_amd64.whl.metadata (15 kB)
  Using cached torch-2.8.0-cp39-cp39-win_amd64.whl.metadata (30 kB)
  Using cached torchvision-0.23.0-cp39-cp39-win_amd64.whl.metadata (6.1 kB)
  Using cached torchaudio-2.8.0-cp39-cp39-win_amd64.whl.metadata (7.2 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.13.1-cp39-cp39-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached filelock-3.19.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.2.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached jinja2-3.1.6-py3-none-any.wh

In [124]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

In [114]:
train_df = pd.read_csv('train_format1.csv')
test_df = pd.read_csv('test_format1.csv')

In [115]:
train_df.head()

,user_id,merchant_id,label
0,34176,3906,0
1,34176,121,0
2,34176,4356,1
3,34176,2217,0
4,230784,4818,0


In [116]:
test_df.head()

,user_id,merchant_id,prob
0,163968,4605,NaN
1,360576,1581,NaN
2,98688,1964,NaN
3,98688,3645,NaN
4,295296,3361,NaN


In [117]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260864 entries, 0 to 260863
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   user_id      260864 non-null  int64
 1   merchant_id  260864 non-null  int64
 2   label        260864 non-null  int64
dtypes: int64(3)
memory usage: 6.0 MB


In [118]:
train_df.isnull().sum()

user_id        0
merchant_id    0
label          0
dtype: int64

In [119]:
X = train_df[['user_id', 'merchant_id']]
y = train_df['label']
X_test_final = test_df[['user_id', 'merchant_id']]

In [120]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [121]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [122]:
lr_pred = lr_model.predict(X_val)
lr_pred_proba = lr_model.predict_proba(X_val)[:, 1]

In [125]:
print("Logistic Regression Results:")
print(f"Accuracy: {accuracy_score(y_val, lr_pred):.4f}")
print(f"Precision: {precision_score(y_val, lr_pred):.4f}")
print(f"Recall: {recall_score(y_val, lr_pred):.4f}")
print(f"F1-Score: {f1_score(y_val, lr_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_val, lr_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, lr_pred))

Logistic Regression Results:
Accuracy: 0.9397
Precision: 0.0000
Recall: 0.0000
F1-Score: 0.0000
ROC-AUC: 0.4945

Classification Report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97     49026
           1       0.00      0.00      0.00      3147

    accuracy                           0.94     52173
   macro avg       0.47      0.50      0.48     52173
weighted avg       0.88      0.94      0.91     52173



In [126]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [127]:
rf_pred = rf_model.predict(X_val)
rf_pred_proba = rf_model.predict_proba(X_val)[:, 1]

In [128]:
print("Random Forest Results:")
print(f"Accuracy: {accuracy_score(y_val, rf_pred):.4f}")
print(f"Precision: {precision_score(y_val, rf_pred):.4f}")
print(f"Recall: {recall_score(y_val, rf_pred):.4f}")
print(f"F1-Score: {f1_score(y_val, rf_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_val, rf_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, rf_pred))

Random Forest Results:
Accuracy: 0.9191
Precision: 0.0991
Recall: 0.0423
F1-Score: 0.0593
ROC-AUC: 0.5280

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.98      0.96     49026
           1       0.10      0.04      0.06      3147

    accuracy                           0.92     52173
   macro avg       0.52      0.51      0.51     52173
weighted avg       0.89      0.92      0.90     52173



In [129]:
class GNNModel(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=128, output_dim=2):
        super(GNNModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)
        return x

In [130]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X.values)
        self.y = torch.LongTensor(y.values)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [131]:
train_dataset = TabularDataset(X_train, y_train)
val_dataset = TabularDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [132]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gnn_model = GNNModel(input_dim=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(gnn_model.parameters(), lr=0.001)

In [133]:
num_epochs = 20
for epoch in range(num_epochs):
    gnn_model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = gnn_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss/len(train_loader):.4f}")

Epoch 5/20, Loss: 0.2568
Epoch 10/20, Loss: 0.2511
Epoch 15/20, Loss: 0.2618
Epoch 20/20, Loss: 0.2307


In [134]:
gnn_model.eval()
gnn_predictions = []
gnn_predictions_proba = []
with torch.no_grad():
    for X_batch, _ in val_loader:
        X_batch = X_batch.to(device)
        outputs = gnn_model(X_batch)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs, 1)
        gnn_predictions.extend(predicted.cpu().numpy())
        gnn_predictions_proba.extend(probs[:, 1].cpu().numpy())

In [135]:
print("GNN Results:")
print(f"Accuracy: {accuracy_score(y_val, gnn_predictions):.4f}")
print(f"Precision: {precision_score(y_val, gnn_predictions):.4f}")
print(f"Recall: {recall_score(y_val, gnn_predictions):.4f}")
print(f"F1-Score: {f1_score(y_val, gnn_predictions):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_val, gnn_predictions_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, gnn_predictions))

GNN Results:
Accuracy: 0.9397
Precision: 0.0000
Recall: 0.0000
F1-Score: 0.0000
ROC-AUC: 0.5000

Classification Report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97     49026
           1       0.00      0.00      0.00      3147

    accuracy                           0.94     52173
   macro avg       0.47      0.50      0.48     52173
weighted avg       0.88      0.94      0.91     52173



In [136]:
class DeepFM(nn.Module):
    def __init__(self, input_dim=2, embed_dim=10, hidden_dims=[128, 64]):
        super(DeepFM, self).__init__()
        
        self.input_dim = input_dim
        self.embed_dim = embed_dim
        
        self.linear = nn.Linear(input_dim, 1)
        
        self.embeddings = nn.Parameter(torch.randn(input_dim, embed_dim))
        
        deep_layers = []
        in_dim = input_dim * embed_dim
        for hidden_dim in hidden_dims:
            deep_layers.append(nn.Linear(in_dim, hidden_dim))
            deep_layers.append(nn.ReLU())
            deep_layers.append(nn.Dropout(0.3))
            in_dim = hidden_dim
        deep_layers.append(nn.Linear(in_dim, 1))
        self.deep = nn.Sequential(*deep_layers)
        
    def forward(self, x):
        linear_part = self.linear(x)
        
        embed = x.unsqueeze(2) * self.embeddings.unsqueeze(0)
        
        sum_squared = torch.sum(embed, dim=1) ** 2
        squared_sum = torch.sum(embed ** 2, dim=1)
        fm_part = 0.5 * torch.sum(sum_squared - squared_sum, dim=1, keepdim=True)
        
        deep_input = embed.view(x.size(0), -1)
        deep_part = self.deep(deep_input)
        
        output = linear_part + fm_part + deep_part
        return output

In [137]:
deepfm_model = DeepFM(input_dim=2).to(device)
criterion_deepfm = nn.BCEWithLogitsLoss()
optimizer_deepfm = optim.Adam(deepfm_model.parameters(), lr=0.001)

In [138]:
num_epochs = 20
for epoch in range(num_epochs):
    deepfm_model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.float().to(device)
        
        optimizer_deepfm.zero_grad()
        outputs = deepfm_model(X_batch).squeeze()
        loss = criterion_deepfm(outputs, y_batch)
        loss.backward()
        optimizer_deepfm.step()
        
        train_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss/len(train_loader):.4f}")

Epoch 5/20, Loss: 390497.5422
Epoch 10/20, Loss: 395972.3272
Epoch 15/20, Loss: 409090.0374
Epoch 20/20, Loss: 421163.7896


In [139]:
deepfm_model.eval()
deepfm_predictions = []
deepfm_predictions_proba = []
with torch.no_grad():
    for X_batch, _ in val_loader:
        X_batch = X_batch.to(device)
        outputs = deepfm_model(X_batch).squeeze()
        probs = torch.sigmoid(outputs)
        predicted = (probs > 0.5).long()
        deepfm_predictions.extend(predicted.cpu().numpy())
        deepfm_predictions_proba.extend(probs.cpu().numpy())

In [140]:
print("DeepFM Results:")
print(f"Accuracy: {accuracy_score(y_val, deepfm_predictions):.4f}")
print(f"Precision: {precision_score(y_val, deepfm_predictions):.4f}")
print(f"Recall: {recall_score(y_val, deepfm_predictions):.4f}")
print(f"F1-Score: {f1_score(y_val, deepfm_predictions):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_val, deepfm_predictions_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_val, deepfm_predictions))

DeepFM Results:
Accuracy: 0.0648
Precision: 0.0604
Recall: 0.9971
F1-Score: 0.1140
ROC-AUC: 0.5009

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.00      0.01     49026
           1       0.06      1.00      0.11      3147

    accuracy                           0.06     52173
   macro avg       0.51      0.50      0.06     52173
weighted avg       0.91      0.06      0.02     52173



In [141]:
results = {
    'Model': ['Logistic Regression', 'Random Forest', 'GNN', 'DeepFM'],
    'Accuracy': [
        accuracy_score(y_val, lr_pred),
        accuracy_score(y_val, rf_pred),
        accuracy_score(y_val, gnn_predictions),
        accuracy_score(y_val, deepfm_predictions)
    ],
    'F1-Score': [
        f1_score(y_val, lr_pred),
        f1_score(y_val, rf_pred),
        f1_score(y_val, gnn_predictions),
        f1_score(y_val, deepfm_predictions)
    ],
    'ROC-AUC': [
        roc_auc_score(y_val, lr_pred_proba),
        roc_auc_score(y_val, rf_pred_proba),
        roc_auc_score(y_val, gnn_predictions_proba),
        roc_auc_score(y_val, deepfm_predictions_proba)
    ]
}

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1-Score', ascending=False)
print("\nModel Comparison:")
print(results_df)


Model Comparison:
                 Model  Accuracy  F1-Score   ROC-AUC
3               DeepFM  0.064842  0.113972  0.500942
1        Random Forest  0.919058  0.059256  0.528037
0  Logistic Regression  0.939681  0.000000  0.494457
2                  GNN  0.939681  0.000000  0.500000


In [142]:
best_model_name = results_df.iloc[0]['Model']
print(f"\nBest Model: {best_model_name}")


Best Model: DeepFM
